# 06 - Config-Driven and Transform

> **When to use**: When you need to reuse configs, batch-generate multi-table data, or transform data after generation.
>
> **Core concept**: Pydantic config models + YAML/JSON files + Transform scripts + Snapshot.

## Applicable Scenarios

- Multi-table batch generation → YAML config + `fill_from_config()`
- Complex business logic (e.g., conditional computation) → Transform Script
- CI/CD reproducible test data → Snapshot + `replay()`
- Team-shared test environment → YAML config committed to Git

## What You Will Learn

- Config model hierarchy: GeneratorConfig → TableConfig → ColumnConfig
- YAML/JSON config formats
- Transform Scripts business logic
- ColumnAssociation cross-table association
- SnapshotManager snapshot management

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| **→ 06** | **Config-Driven and Transform** | **Config / Core** | **01** |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---
## Setup

Use Python 3.10+ and run this notebook from `examples/notebooks` in a repository checkout. Select a notebook kernel from the environment containing these packages:

```bash
python -m pip install 'sqlseed[mimesis]==0.2.4' 'sqlseed-cli==0.2.4' jupyterlab
```

For source development, install Core and CLI together as described in the [repository README](../../README.md). This notebook creates its own temporary database and cache. Run cells from top to bottom; the validation helpers raise on partial generation or failed CLI commands.


In [ ]:
ORG_QUERY = "SELECT org_code FROM organizations"
ORG_PATTERN = "ORG-\\d{4}"
from sqlseed.config.models import GeneratorConfig, TableConfig, ColumnConfig, ProviderType, ColumnConstraintsConfig, ColumnAssociation
from sqlseed.config.loader import save_config, load_config, generate_template
from sqlseed.config.snapshot import SnapshotManager
from pathlib import Path

import sqlite3
# Install the packages listed in Setup into the selected notebook kernel.
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
import os
import tempfile
from pathlib import Path
notebook_temp = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-06-")
work_dir = Path(notebook_temp.name)
os.environ["SQLSEED_CACHE_DIR"] = str(work_dir / "cache")
db_path = build(work_dir / "demo.db")

# Fail visibly if a generation only partially succeeds.
generation_checks = []
def check_result(result, expected_count):
    if result.errors or result.count != expected_count:
        raise RuntimeError(f"{result.table_name}: expected {expected_count}, wrote {result.count}; errors={result.errors}")
    generation_checks.append({"table": result.table_name, "count": result.count, "errors": list(result.errors)})
    print(f"Verified {result.table_name}: {result.count} rows; errors={result.errors}")
    return result

def check_results(results, config_path):
    config = sqlseed.load_config(str(config_path))
    expected = {table.name: table.count for table in config.tables}
    for result in results:
        check_result(result, expected[result.table_name])
    if {result.table_name for result in results} != set(expected):
        raise RuntimeError("Not every configured table produced a result")
    return results

def check_cli(result):
    if result.exit_code != 0:
        raise RuntimeError(result.output) from result.exception
    return result


# Populate base dependencies
with connect(str(db_path)) as orch:
    check_result(orch.fill_table("organizations", count=5, seed=42), 5)
    check_result(orch.fill_table("members", count=20, seed=42), 20)
    check_result(orch.fill_table("projects", count=10, seed=42), 10)
    check_result(orch.fill_table("tags", count=8, seed=42), 8)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Transform Loading | `src/sqlseed/core/transform.py` | `TransformLoader` |
| Config Model | `src/sqlseed/config/models.py` | `GeneratorConfig` |

> Corresponding architecture diagram: [§9 Config Model Hierarchy](../../docs/architecture.zh-CN.md#9-配置模型层次结构)

## 1. See It in Action — YAML-Driven Batch Filling

For complex multi-table scenarios, declare all tables and columns in a YAML config file, then batch-fill with one line of code:

```yaml
db_path: "app.db"
tables:
  - name: users
    count: 10000
    columns:
      - name: email
        generator: email
```
The executable example below appends to the initialized fixture. Use a fresh database or clear children before parents when resetting related data.


In [ ]:

# Build config with Python objects (equivalent to YAML)
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    tables=[
        TableConfig(name="organizations", count=5),
        TableConfig(name="members", count=10),
    ]
)
config_path = (work_dir / "_demo_config.yaml")
save_config(config, str(config_path))

# One line to batch-fill multiple tables
results = check_results(fill_from_config(str(config_path)), str(config_path))
print(f"{'Table':<15s}  {'Rows':>6s}  {'Time':>8s}  {'Speed':>10s}")
print('-' * 45)
for r in results:
    print(f"{r.table_name:<15s}  {r.count:>6d}  {r.elapsed:>7.3f}s  {r.rows_per_second:>8.0f} rows/s")

config_path.unlink(missing_ok=True)

Benefits of config-driven approach:

- **Version controllable** — YAML files can be committed to Git
- **Reproducible** — Fixed config and seed support reproducibility when schema, existing data and provider versions also match
- **Shareable** — Team members use the same config

Below we break down each layer of the config model.

## 2. Config Model Hierarchy

sqlseed uses Pydantic models to define the config hierarchy:

```
GeneratorConfig
├── db_path or url (exactly one)
├── provider: ProviderType (MIMESIS)
├── locale: str (en_US)
├── tables: list[TableConfig]
│   └── TableConfig
│       ├── name: str
│       ├── count: int (1000)
│       ├── columns: list[ColumnConfig]
│       │   └── ColumnConfig
│       │       ├── Source mode: generator + params
│       │       └── Derived mode: derive_from + expression
│       └── clear_before, seed, transform, enrich
├── associations: list[ColumnAssociation]
└── optimize_pragma, snapshot_dir (log_level is deprecated)
```

In [ ]:
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    locale="en_US",
    tables=[
        TableConfig(
            name="organizations",
            count=5,
            columns=[
                ColumnConfig(name="name", generator="company"),
                ColumnConfig(name="description", generator="sentence"),
            ],
        ),
    ],
)
print(f"GeneratorConfig: db_path={Path(config.db_path).name}")
print(f"  provider={config.provider}, locale={config.locale}")
print(f"  tables: {[t.name for t in config.tables]}")
print(f"  TableConfig[0]: count={config.tables[0].count}, columns={len(config.tables[0].columns)}")

## 3. ColumnConfig Dual-Mode Validation

ColumnConfig has two mutually exclusive modes:
- **Source mode**: `generator` + `params` + `null_ratio` + `provider`
- **Derived mode**: `derive_from` + `expression`

Pydantic `model_validator` enforces mutual exclusion; setting both `generator` and `derive_from` raises an error.

In [ ]:
try:
    bad_config = ColumnConfig(
        name="test",
        generator="email",
        derive_from="other_col",
    )
except Exception as e:
    print(f"❌ Dual-mode conflict: {type(e).__name__}")
    print(f"   {e}")

print("\n✅ Source mode (generator):")
src = ColumnConfig(name="email_col", generator="email")
print(f"   generator={src.generator}, derive_from={src.derive_from}")

print("\n✅ Derived mode (derive_from):")
drv = ColumnConfig(name="short_code", derive_from="project_no", expression="value[-6:]")
print(f"   generator={drv.generator}, derive_from={drv.derive_from}, expression={drv.expression}")

## 4. Complete YAML Config in Practice

In [ ]:
config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType.MIMESIS,
    tables=[
        TableConfig(name="organizations", count=5),
        TableConfig(
            name="members",
            count=20,
            columns=[
                ColumnConfig(name="member_no", generator="pattern", params={"regex": "M-\\d{6}"}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
                ColumnConfig(name="email", generator="email", constraints=ColumnConstraintsConfig(unique=True)),
            ],
        ),
        TableConfig(
            name="projects",
            count=10,
            columns=[
                ColumnConfig(name="project_no", generator="pattern", params={"regex": "PRJ-\\d{6}"}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
            ],
        ),
    ],
)
config_path = (work_dir / "_demo_config.yaml")
save_config(config, str(config_path))

results = check_results(fill_from_config(str(config_path)), str(config_path))
for r in results:
    status = "✅" if r.count > 0 else "❌"
    print(f"{status} {r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

## 5. JSON Config Format

`save_config` and `load_config` support both YAML and JSON formats, auto-detected by file extension.

In [ ]:
import json

json_path = (work_dir / "_demo_config.json")

save_config(config, str(json_path))
with open(json_path) as f:
    data = json.load(f)
print("JSON config format:")
safe = json.dumps(data, indent=2, ensure_ascii=False)
print(safe[:500] + "...")

json_path.unlink(missing_ok=True)

## 6. load_config Demo

In [ ]:
loaded = load_config(str(config_path))
print("load_config result:")
print(f"  db_path: {Path(loaded.db_path).name}")
print(f"  provider: {loaded.provider}")
print(f"  tables: {[t.name for t in loaded.tables]}")
print(f"  tables[1].columns: {[c.name for c in loaded.tables[1].columns]}")

## 7. generate_template / sqlseed init

`generate_template()` auto-generates a config template based on DB schema; the CLI equivalent is `sqlseed init`.

In [ ]:
template = generate_template(str(db_path), table_name="organizations")
print("generate_template result:")
print(f"  db_path: {Path(template.db_path).name}")
print(f"  tables: {[t.name for t in template.tables]}")
if template.tables:
    t = template.tables[0]
    print(f"  '{t.name}' columns ({len(t.columns)}):")
    for c in t.columns[:5]:
        print(f"    - {c.name}: generator={c.generator}")

## 8. Transform Scripts — Complex Business Logic

For business logic that declarative configs cannot express, use a Python Transform Script. The script must define a `transform_row(row, ctx)` function that transforms each row after generation and before writing to the database.

**Typical scenarios**:
- Conditional computation (e.g., compute VIP level by age)
- Data formatting (e.g., add international prefix to phone numbers)
- Field composition (e.g., concatenate full_name)
- Data validation and correction

In [ ]:
transform_script = (work_dir / "_demo_transform.py")
transform_script.write_text(
    "def transform_row(row, ctx):\n"
    "    # Compute organization size tier by member_count\n"
    "    count = row.get('member_count', 0) or 0\n"
    "    if count >= 200:\n"
    '        row[\'description\'] = f"[Large] {row.get(\'name\', \'\')} - Global leading tech enterprise"\n'
    "    elif count >= 50:\n"
    '        row[\'description\'] = f"[Medium] {row.get(\'name\', \'\')} - Fast-growing tech company"\n'
    "    else:\n"
    '        row[\'description\'] = f"[Small] {row.get(\'name\', \'\')} - Innovative startup"\n'
    "    # Uppercase the name\n"
    "    if row.get('name'):\n"
    "        row['name'] = row['name'].upper()\n"
    "    return row\n"
)

with connect(str(db_path)) as orch:
    result = check_result(orch.fill_table("organizations", count=5, transform=str(transform_script),
        columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                 "parent_code": {"type": "choice", "choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute(ORG_QUERY).fetchall()]]}}), 5)
    print(f"Transform fill: {result.count} rows")

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT name, member_count, description FROM organizations ORDER BY rowid DESC LIMIT 5").fetchall()
for r in rows:
    assert r[0] == r[0].upper()
    expected_tier = "Large" if r[1] >= 200 else "Medium" if r[1] >= 50 else "Small"
    assert r[2].startswith(f"[{expected_tier}] ")
    desc = str(r[2])[:60]
    print(f"  {r[0]:<20s} | members={r[1]:4d} | {desc}")
conn.close()

transform_script.unlink(missing_ok=True)


## 9. ColumnAssociation Cross-Table Association

`ColumnAssociation` declares that multiple tables share the same column values, ensuring FK reference consistency.

In [ ]:
config_with_assoc = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name="organizations", count=3, columns=[
            ColumnConfig(name="org_code", generator="pattern", params={"regex": ORG_PATTERN}),
            ColumnConfig(name="parent_code", generator="choice", params={"choices": [*[r[0] for r in __import__("sqlite3").connect(str(db_path)).execute(ORG_QUERY).fetchall()]]}),
        ]),
        TableConfig(name="members", count=10),
    ],
    associations=[
        ColumnAssociation(
            column_name="org_code",
            source_table="organizations",
            target_tables=["members"],
            strategy="shared_pool",
        ),
    ],
)
assoc_path = (work_dir / "_assoc_config.yaml")
save_config(config_with_assoc, str(assoc_path))

results = check_results(fill_from_config(str(assoc_path)), str(assoc_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows")

conn = sqlite3.connect(str(db_path))
org_codes = [r[0] for r in conn.execute("SELECT DISTINCT org_code FROM organizations").fetchall()]
member_orgs = [r[0] for r in conn.execute("SELECT org_code FROM members LIMIT 5").fetchall()]
print(f"\nAssociation validation: org_codes={org_codes}")
print(f"  members org_code: {member_orgs}")
all_valid = all(m in org_codes for m in member_orgs if m)
assert all_valid
print(f"  All FK references valid: {all_valid}")
conn.close()

assoc_path.unlink(missing_ok=True)


## 10. ColumnConstraintsConfig Constraint Configuration

`ColumnConstraintsConfig` supports the following constraints:
- `unique`: unique constraint
- `min_value` / `max_value`: numeric range
- `regex`: regex matching
- `max_retries`: max retry count

In [ ]:
constrained = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(
            name="organizations",
            count=5,
            columns=[
                ColumnConfig(name="name", generator="company", constraints=ColumnConstraintsConfig(unique=True)),
                ColumnConfig(name="org_code", generator="pattern", params={"regex": ORG_PATTERN}, constraints=ColumnConstraintsConfig(unique=True)),  # noqa: E501
            ],
        ),
    ],
)
c_path = (work_dir / "_constraints_config.yaml")
save_config(constrained, str(c_path))

results = check_results(fill_from_config(str(c_path)), str(c_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows")

conn = sqlite3.connect(str(db_path))
names = [r[0] for r in conn.execute("SELECT name FROM organizations").fetchall()]
codes = [r[0] for r in conn.execute(ORG_QUERY).fetchall()]
print(f"\nUNIQUE name: {len(names) == len(set(names))}")
print(f"UNIQUE org_code: {len(codes) == len(set(codes))}")
conn.close()

c_path.unlink(missing_ok=True)
config_path.unlink(missing_ok=True)

## 11. SnapshotManager Configuration Snapshots

SnapshotManager saves, loads and lists generation configurations. It does not expose `replay()` and does not back up database rows. The CLI `sqlseed replay` executes the stored config. Replay honors `clear_before`; this example appends three rows, and IDs may differ from prior runs.

In [ ]:
from click.testing import CliRunner
from sqlseed_cli import cli

snap_mgr = SnapshotManager(work_dir / "snapshots")
config = GeneratorConfig(
    db_path=str(db_path),
    tables=[TableConfig(name="organizations", count=3, seed=42)],
)
snapshot_path = snap_mgr.save(config, "organizations", count=3, seed=42)
print("Snapshot saved:", snapshot_path)
assert len(snap_mgr.list_snapshots()) == 1
data = snap_mgr.load(snapshot_path)
print("Saved table/count:", data["table_name"], data["count"])

with sqlite3.connect(str(db_path)) as connection:
    before_replay = connection.execute("SELECT count(*) FROM organizations").fetchone()[0]
replay_result = check_cli(CliRunner().invoke(cli, ["replay", str(snapshot_path)]))
print(replay_result.output)
with sqlite3.connect(str(db_path)) as connection:
    after_replay = connection.execute("SELECT count(*) FROM organizations").fetchone()[0]
assert after_replay == before_replay + 3, (before_replay, after_replay)
print("Replay appended:", after_replay - before_replay)


### CLI Snapshot Commands

For an existing independent `app.db` with a `users` table:

```bash
sqlseed fill app.db --table users --count 100 --seed 42 --snapshot
# The command prints <cache_dir>/snapshots/YYYY-MM-DD_HHMMSS_ffffff_users.yaml.
# Substitute that actual path below.
sqlseed replay "/path/to/saved-snapshot.yaml"
```

Use snapshots to reuse reviewed generation settings. They do not restore an earlier database state.

## 12. Preview & Debug CLI

`sqlseed preview` previews data without writing; `sqlseed inspect --show-mapping` shows column mapping strategies.

In [ ]:
from click.testing import CliRunner

from sqlseed_cli import cli

preview_rows = preview(str(db_path), table="organizations", count=3,
                       columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                                "name": {"type": "company"}})
print("preview result:")
for row in preview_rows:
    print(f"  {row.get('org_code', 'N/A')} | {row.get('name', 'N/A')}")



runner = CliRunner()
result = check_cli(runner.invoke(cli, ["inspect", str(db_path), "-t", "organizations", "--show-mapping"]))
if result.output.strip():
    print("\nsqlseed inspect --show-mapping:")
    print(result.output[:500])

## ✅ Summary

| Feature | API | Status |
|---|---|---|
| Config Model Hierarchy | GeneratorConfig/TableConfig/ColumnConfig | ✅ |
| Dual-Mode Validation | ColumnConfig model_validator | ✅ |
| YAML/JSON Config | save_config/load_config | ✅ |
| Auto Template Generation | generate_template / sqlseed init | ✅ |
| Transform Scripts | transform_row(row, ctx) | ✅ |
| Cross-Table Association | ColumnAssociation | ✅ |
| Constraint Config | ColumnConstraintsConfig | ✅ |
| Snapshot Management | SnapshotManager | ✅ |
| Preview & Debug | preview / inspect --show-mapping | ✅ |

**Next**: [07-ai-plugin.ipynb](07-ai-plugin.ipynb) — AI Smart Config

In [ ]:
# Verify exact database totals after the full notebook, plus every declared FK.
import sqlite3
expected_counts = {'organizations': 31, 'members': 60, 'projects': 20, 'tasks': 0, 'tags': 8, 'reviews': 0}
with sqlite3.connect(str(db_path)) as verification_db:
    actual_counts = {
        table: verification_db.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
        for table in expected_counts  # Fixed tutorial table names.
    }
    assert actual_counts == expected_counts, (actual_counts, expected_counts)
    fk_errors = verification_db.execute("PRAGMA foreign_key_check").fetchall()
    assert not fk_errors, fk_errors
print("Verified database row counts:", actual_counts)
print("Database FK check:", fk_errors)
print("Verified fill operations:", len(generation_checks))
